# Module 4 Lab: See the Unseen
## 6.3710x — Probability and Statistical Data Analysis

### Where We Left Off

In Module 3, you built covariance matrices in 2D and 3D, computed eigendecompositions, and learned that eigenvectors reveal the natural axes of variation while eigenvalues tell you how much variance lives along each axis. At the very end, you saw something you couldn't explain: ratio-colored scatter plots showed **banding** that sex alone could not account for. There was a hidden variable.

### What's Changed

1. **All five measurements.** No more choosing subsets — you'll work with FL, RW, CL, CW, and BD simultaneously.
2. **Dimensionality reduction.** You can't plot 5D directly. But eigendecomposition will find the axes that matter most and project the data down to 2D.
3. **Uncertainty quantification.** You'll ask: how certain are we about the structure we see?
4. **Model selection.** You'll formalize "one population or two?" using the Akaike Information Criterion.

### The Plan

We'll start where Module 3 left off — with the mystery. You'll apply PCA to all five measurements, and the hidden variable will reveal itself. Then you'll interrogate the result using bootstrap, confidence intervals, and model selection.

### What's New Conceptually

- **PCA** — eigendecomposition of the 5×5 covariance matrix, projection onto the top principal components
- **Bootstrap** — resampling from the empirical distribution to approximate sampling distributions
- **Confidence intervals** — Normal approximation vs. bootstrap percentiles, and when each is appropriate
- **Model selection with AIC** — rewarding better fit while penalizing complexity


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11

np.random.seed(42)

def load_data(filename):
    """Load CSV from local directory or GitHub."""
    if os.path.exists(filename):
        return pd.read_csv(filename)
    base_url = ("https://raw.githubusercontent.com/"
                "codey-m/prob-stats-labs/main/")
    url = base_url + filename
    print(f"Loading from GitHub: {filename}")
    return pd.read_csv(url)

## Part 1: From 2D to 5D — The Full Dataset

Let's load the full crab dataset and see the challenge in front of us.


In [ ]:
crabs = load_data("crabs.csv")

print("Shape:", crabs.shape)
print("Columns:", list(crabs.columns))
crabs.head()

In [ ]:
measurement_cols = ["FL", "RW", "CL", "CW", "BD"]
crabs[measurement_cols].describe().round(2)

The dataset contains 200 crabs. Each row is one crab; each column is a measured feature. You also have two categorical variables: `sex` and `sp` (species). We'll leave `sp` hidden for a little longer.

### TODO 1a

Extract the five measurement columns into a NumPy array $X$ with shape $(n, 5)$.


In [ ]:
# TODO 1a: Build the data matrix
X = ...

print(f"Shape of X: {X.shape}")
n = X.shape[0]

### TODO 1b

Center the columns of $X$ and compute the $5 \times 5$ sample covariance matrix using matrix multiplication.

Recall the formula:

$$
S = \frac{1}{n-1} X_c^T X_c
$$

where $X_c$ is the centered data matrix.


In [ ]:
# TODO 1b: Center and compute covariance
X_mean = ...
X_centered = ...
cov_matrix = ...

print(f"Covariance matrix shape: {cov_matrix.shape}")
print("\nCovariance matrix:")
print(np.round(cov_matrix, 2))

In [ ]:
# Checkpoint
cov_check = np.cov(X, rowvar=False)
assert np.allclose(cov_matrix, cov_check), \
    "Your covariance matrix does not match np.cov."
print("✓ Covariance matrix is correct.")

### TODO 1c

Compute the correlation matrix from your covariance matrix.

Recall:

$$
\rho_{ij} = \frac{\operatorname{Cov}(X_i, X_j)}
{\sqrt{\operatorname{Var}(X_i)\operatorname{Var}(X_j)}}
$$


In [ ]:
# TODO 1c: Correlation matrix
stds = ...
cor_matrix = ...

print("Correlation matrix:")
print(np.round(cor_matrix, 3))

In [ ]:
# Provided: heatmap visualization
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cor_matrix, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels(measurement_cols)
ax.set_yticklabels(measurement_cols)

for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cor_matrix[i, j]:.2f}",
                ha="center", va="center", fontsize=10)

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation Matrix — All Five Measurements",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**What to notice:**

All five measurements are highly correlated. Bigger crabs are bigger in every dimension. But you can't look at this matrix and directly *see* the structure in 5D.

This is the new problem of Module 4:

> **What does the covariance structure tell us in five dimensions, and how do we visualize something we cannot see directly?**

The answer is PCA.

---


## Part 2: PCA Mechanics — The Same Idea, Bigger Matrix

In Module 3, you eigendecomposed 2×2 and 3×3 covariance matrices. PCA does the same thing here in 5D.

The key steps are:

1. eigendecompose the covariance matrix
2. sort the eigenpairs from largest to smallest eigenvalue
3. interpret eigenvalues as variance explained
4. project the data onto the top eigenvectors


### TODO 2a

Compute the eigenvalues and eigenvectors of the covariance matrix using `np.linalg.eigh()`. Then sort them in descending order.


In [ ]:
# TODO 2a: Eigendecomposition and sorting
eigenvalues_raw, eigenvectors_raw = ...

sort_idx = ...
eigenvalues = ...
eigenvectors = ...

print("Sorted eigenvalues (descending):")
for i, val in enumerate(eigenvalues):
    print(f"  λ_{i+1} = {val:.4f}")

### TODO 2b

Compute the proportion of variance explained (PVE) by each principal component and the cumulative PVE.

$$
\text{PVE}_k = \frac{\lambda_k}{\sum_{j=1}^p \lambda_j}
$$


In [ ]:
# TODO 2b: Variance explained
pve = ...
cpve = ...

print("Variance explained by each PC:")
for i in range(5):
    print(f"  PC{i+1}: {pve[i]:.4f}  ({pve[i]*100:.1f}%)"
          f"    Cumulative: {cpve[i]*100:.1f}%")

In [ ]:
# Provided: scree plot
fig, ax = plt.subplots(figsize=(7, 4))

pc_idx = np.arange(1, 6)
ax.bar(pc_idx, pve, color="steelblue", alpha=0.75,
       edgecolor="white", linewidth=1.5,
       label="Individual PVE")
ax.plot(pc_idx, cpve, color="darkorange", marker="o",
        linewidth=2.5, label="Cumulative PVE")
ax.axhline(0.95, color="gray", linestyle="--",
           linewidth=1.2, alpha=0.7, label="95% threshold")

ax.set_xlabel("Principal Component")
ax.set_ylabel("Proportion of Variance Explained")
ax.set_title("Scree Plot", fontsize=13, fontweight="bold")
ax.set_xticks(pc_idx)
ax.set_xticklabels([f"PC{i}" for i in pc_idx])
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
pcs_for_95 = np.argmax(cpve >= 0.95) + 1
print(f"Number of PCs needed to reach 95% variance: {pcs_for_95}")

Let's reflect on this:

- How much of the total variance is captured by PC1 alone?
- How many PCs do you need to retain to capture at least 95% of the variance?
- Does the data look effectively low-dimensional, or does it spread meaningfully across many directions?


### TODO 2c

Project the centered data onto the first two principal components.

If $V_2$ is the $(5 \times 2)$ matrix whose columns are the top two eigenvectors, then the projected data is

$$
Z = X_c V_2
$$


In [ ]:
# TODO 2c: Projection onto PC1 and PC2
V2 = ...
Z = ...

print(f"Shape of Z: {Z.shape}")
print("First five projected points:")
print(np.round(Z[:5], 3))

In [ ]:
# Provided: PC1 vs PC2 colored by sex
fig, ax = plt.subplots(figsize=(8, 6))

for sex, color, marker in [("M", "steelblue", "o"),
                           ("F", "coral", "s")]:
    mask = crabs["sex"] == sex
    ax.scatter(Z[mask, 0], Z[mask, 1],
               c=color, marker=marker,
               alpha=0.6, s=35,
               edgecolor="white", linewidth=0.3,
               label=sex)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Projection onto PC1 and PC2 — Colored by Sex",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**What to notice:**

You should see some separation by sex, but not enough to explain the full structure. There are **two clouds** or bands that sex alone does not account for. This is the same mystery you noticed at the end of Module 3.

### TODO 2d

Inspect the loadings of PC1 and PC2 — the components of the first two eigenvectors.


In [ ]:
# TODO 2d: Loadings
loadings = pd.DataFrame({
    "measurement": measurement_cols,
    "PC1": eigenvectors[:, 0],
    "PC2": eigenvectors[:, 1]
})
loadings

In [ ]:
# Provided: loading plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].bar(loadings["measurement"], loadings["PC1"],
            color="steelblue", edgecolor="white")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("PC1 Loadings", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Loading")

axes[1].bar(loadings["measurement"], loadings["PC2"],
            color="darkorange", edgecolor="white")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("PC2 Loadings", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Let's reflect on this:

1. Do the PC1 loadings all have the same sign, or do some point in opposite directions? What does that suggest PC1 is measuring biologically?
2. Which measurements contribute most strongly to PC2? What kind of shape contrast might PC2 represent?
3. Why do the top eigenvectors give the "best" 2D projection for preserving variance?


## Part 3: The Reveal

Now we reveal the hidden variable.


In [ ]:
# Provided: compare colorings by sex and species
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sex
for sex, color, marker in [("M", "steelblue", "o"),
                           ("F", "coral", "s")]:
    mask = crabs["sex"] == sex
    axes[0].scatter(Z[mask, 0], Z[mask, 1],
                    c=color, marker=marker,
                    alpha=0.6, s=35,
                    edgecolor="white", linewidth=0.3,
                    label=sex)
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].set_title("Colored by Sex",
                  fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Right: species
for sp, color, label in [("B", "dodgerblue", "Blue"),
                         ("O", "darkorange", "Orange")]:
    mask = crabs["sp"] == sp
    axes[1].scatter(Z[mask, 0], Z[mask, 1],
                    c=color,
                    alpha=0.6, s=35,
                    edgecolor="white", linewidth=0.3,
                    label=label)
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].set_title("Colored by Species",
                  fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle("The Hidden Variable Revealed",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

**The hidden variable is species.**

The five body measurements encode species differences so strongly that the first principal component separates Blue from Orange *Leptograpsus* crabs almost perfectly.

The mystery from Module 3 is now resolved. The banding in the ratio-colored plots, the hints of bimodality from earlier modules — it was species all along.

Let's reflect on this:

1. Why didn't the 2D and 3D analyses in Module 3 reveal this as clearly?
2. PCA found the direction of **maximum variance**. What does it mean that species separation aligns with that direction?
3. Within each species cloud, what might the remaining variation represent?


## Part 4: Quantifying Uncertainty with Bootstrap

We have a striking picture. But how certain are we?

For example:
- PC1 explains a large fraction of the total variance
- the first two PCs capture most of the structure
- the leading eigenvalues seem well separated

Would those conclusions change if we had sampled a different set of crabs?

Bootstrap gives us a computational way to approximate the sampling distribution of a statistic by resampling from the empirical distribution.

We'll first apply it to a simple statistic, then reuse the same pattern for PCA.


### A short bridge from one-dimensional estimation

For a Normal model, plug-in estimation, method of moments, and MLE all agree on the sample mean. We'll use the mean of CL as our warm-up statistic and compute its analytic standard error.

### TODO 4a

Compute the mean of CL, the MLE variance $\hat{\sigma}^2 = \frac{1}{n}\sum (X_i - \bar{X})^2$, and the analytic standard error of the mean using the unbiased sample standard deviation.


In [ ]:
CL = crabs["CL"].values

# TODO 4a: Simple one-dimensional summaries
mean_cl = ...
var_cl_mle = ...
std_cl_unbiased = ...
se_cl = ...

print(f"Mean of CL:            {mean_cl:.3f}")
print(f"MLE variance of CL:    {var_cl_mle:.3f}")
print(f"Analytic SE of mean:   {se_cl:.4f}")

### TODO 4b

Write a bootstrap loop for the mean of CL. This is the first time through the pattern, so you should implement it yourself.

Pattern:
1. resample with replacement
2. recompute the statistic
3. store the result


In [ ]:
# TODO 4b: Bootstrap the mean of CL
n_boot = 5000
boot_means = np.zeros(n_boot)

for b in range(n_boot):
    CL_boot = ...
    boot_means[b] = ...


### TODO 4c

Compute the bootstrap standard error — the standard deviation of the bootstrap means — and compare it to the analytic SE.


In [ ]:
# TODO 4c: Bootstrap SE for mean CL
boot_se_cl = ...

print(f"Analytic SE:   {se_cl:.4f}")
print(f"Bootstrap SE:  {boot_se_cl:.4f}")

In [ ]:
# Provided: bootstrap histogram for the mean
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(boot_means, bins=40, color="steelblue",
        alpha=0.75, edgecolor="white")
ax.axvline(mean_cl, color="darkred", linestyle="--",
           linewidth=2, label=f"Observed mean = {mean_cl:.3f}")

ax.set_xlabel("Bootstrap sample mean of CL")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Distribution of the Sample Mean",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

**What to notice:**

The bootstrap SE should be close to the analytic SE. This is your first confirmation that bootstrap is recovering a sampling distribution you already understand.

Now reuse the same idea for a statistic where there is no simple analytic formula.

### TODO 4d

Bootstrap the **proportion of variance explained by PC1**.

For each bootstrap sample:
1. resample **rows** of $X$ with replacement
2. center the resampled data
3. compute the covariance matrix
4. eigendecompose
5. sort eigenvalues descending
6. compute PVE for PC1

Also record the full vector of eigenvalues for each bootstrap replicate.


In [ ]:
# TODO 4d: Bootstrap PCA statistics
n_boot_pca = 2000
boot_pve1 = np.zeros(n_boot_pca)
boot_eigenvalues = np.zeros((n_boot_pca, 5))

for b in range(n_boot_pca):
    idx = ...
    X_boot = ...
    X_boot_centered = ...
    cov_boot = ...
    evals_boot, _ = ...
    evals_boot = ...
    boot_pve1[b] = ...
    boot_eigenvalues[b] = evals_boot

### TODO 4e

Compute the bootstrap standard error for PVE of PC1.


In [ ]:
# TODO 4e: Bootstrap SE for PVE of PC1
boot_se_pve1 = ...

print(f"Observed PVE for PC1: {pve[0]*100:.2f}%")
print(f"Bootstrap SE:         {boot_se_pve1*100:.2f}%")

In [ ]:
# Provided: bootstrap histogram for PVE of PC1
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(boot_pve1, bins=40, color="darkorange",
        alpha=0.75, edgecolor="white")
ax.axvline(pve[0], color="black", linestyle="--",
           linewidth=2,
           label=f"Observed PVE = {pve[0]:.4f}")

ax.set_xlabel("Proportion of Variance Explained by PC1")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Distribution of PVE for PC1",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### TODO 4f

Visualize the bootstrap distributions of all five eigenvalues.


In [ ]:
# Provided: bootstrap distributions of all eigenvalues
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), sharey=True)

for i in range(5):
    axes[i].hist(boot_eigenvalues[:, i], bins=35,
                 color="steelblue", alpha=0.75,
                 edgecolor="white")
    axes[i].axvline(eigenvalues[i], color="darkred",
                    linestyle="--", linewidth=2)
    axes[i].set_title(f"λ_{i+1}")
    axes[i].set_xlabel("Eigenvalue")

axes[0].set_ylabel("Frequency")
plt.suptitle("Bootstrap Distributions of the Five Eigenvalues",
             fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

Let's reflect on this:

1. Is the bootstrap distribution of $\lambda_1$ roughly symmetric, or is it skewed? What about $\lambda_5$?
2. Relative to their magnitude, are the smaller eigenvalues estimated more precisely or less precisely?
3. What uncertainty are you approximating with bootstrap here, and why would deriving it analytically be difficult?


## Part 5: Confidence Intervals

Bootstrap gives you an approximate sampling distribution. From that, you can estimate uncertainty in several ways.

In this section, you'll compare two interval constructions:

1. **Normal approximation**
2. **Bootstrap percentile interval**

The key question is not just "how do I compute the interval?" but also:

> **When is this interval method appropriate for this statistic?**

**Important interpretation note**

A 95% confidence interval does **not** mean "there is a 95% probability the true parameter lies in this range." The parameter is fixed. The interval is random. The correct frequentist interpretation is:

> If we repeated the data-collection-and-interval-construction procedure many times, about 95% of the resulting intervals would contain the true value.


### Case 1: Mean of CL

This is the friendly case. The sampling distribution of the sample mean is approximately Normal, so the Normal interval should work well.

### TODO 5a

Compute a 95% Normal-approximation confidence interval for the mean of CL:

$$
\hat{\theta} \pm 1.96 \cdot \widehat{\mathrm{se}}
$$


In [ ]:
# TODO 5a: Normal CI for mean CL
ci_mean_normal_lower = ...
ci_mean_normal_upper = ...

print(f"95% CI for mean CL (Normal): "
      f"[{ci_mean_normal_lower:.3f}, {ci_mean_normal_upper:.3f}]")

### TODO 5b

Compute a 95% bootstrap percentile confidence interval for the mean of CL using the 2.5th and 97.5th percentiles of `boot_means`.


In [ ]:
# TODO 5b: Bootstrap percentile CI for mean CL
ci_mean_boot_lower = ...
ci_mean_boot_upper = ...

print(f"95% CI for mean CL (Bootstrap): "
      f"[{ci_mean_boot_lower:.3f}, {ci_mean_boot_upper:.3f}]")

In [ ]:
print(f"Normal CI width:    "
      f"{ci_mean_normal_upper - ci_mean_normal_lower:.3f}")
print(f"Bootstrap CI width: "
      f"{ci_mean_boot_upper - ci_mean_boot_lower:.3f}")

**What to notice:**

These intervals should be very similar. This is a case where the Normal approximation is well justified.

### Case 2: PVE of PC1

Now consider a more complicated statistic: the proportion of variance explained by PC1. This statistic is bounded between 0 and 1, depends nonlinearly on all eigenvalues, and does not have an obvious Gaussian sampling distribution.

### TODO 5c

Compute both a Normal-approximation CI and a bootstrap percentile CI for PVE of PC1.


In [ ]:
# TODO 5c: Confidence intervals for PVE of PC1

# Normal approximation
ci_pve_normal_lower = ...
ci_pve_normal_upper = ...

# Bootstrap percentile
ci_pve_boot_lower = ...
ci_pve_boot_upper = ...

print(f"95% CI for PVE of PC1 (Normal):    "
      f"[{ci_pve_normal_lower:.4f}, {ci_pve_normal_upper:.4f}]")
print(f"95% CI for PVE of PC1 (Bootstrap): "
      f"[{ci_pve_boot_lower:.4f}, {ci_pve_boot_upper:.4f}]")

In [ ]:
# Provided: compare both intervals on the bootstrap histogram
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(boot_pve1, bins=45, density=True,
        color="steelblue", alpha=0.45, edgecolor="white")

ax.axvline(pve[0], color="black", linewidth=2,
           label="Observed value")

ax.axvline(ci_pve_normal_lower, color="darkred",
           linestyle="--", linewidth=2,
           label="Normal CI")
ax.axvline(ci_pve_normal_upper, color="darkred",
           linestyle="--", linewidth=2)

ax.axvline(ci_pve_boot_lower, color="darkgreen",
           linestyle="-", linewidth=2,
           label="Bootstrap CI")
ax.axvline(ci_pve_boot_upper, color="darkgreen",
           linestyle="-", linewidth=2)

ax.set_xlabel("Proportion of Variance Explained by PC1")
ax.set_ylabel("Density")
ax.set_title("Confidence Intervals for PVE of PC1",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

Let's reflect on this:

1. Are the two intervals similar or noticeably different?
2. Does the bootstrap histogram look symmetric enough to support a Normal approximation?
3. Why might the bootstrap percentile interval be more trustworthy here than it was necessary for the sample mean?


### Case 3: The eigenvalue ratio $\lambda_1 / \lambda_2$

This statistic tells you how dominant the first principal component is relative to the second. It combines two uncertain quantities and has no simple analytic standard error formula.

### TODO 5d

Use the bootstrap eigenvalues to compute:
- the observed ratio $\lambda_1 / \lambda_2$
- the bootstrap distribution of that ratio
- a 95% bootstrap percentile CI


In [ ]:
# TODO 5d: Bootstrap CI for eigenvalue ratio
boot_ratio = ...
observed_ratio = ...

ci_ratio_lower = ...
ci_ratio_upper = ...

print(f"Observed λ₁/λ₂: {observed_ratio:.2f}")
print(f"95% Bootstrap CI: [{ci_ratio_lower:.2f}, "
      f"{ci_ratio_upper:.2f}]")

In [ ]:
# Provided: bootstrap histogram for eigenvalue ratio
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(boot_ratio, bins=40, color="mediumpurple",
        alpha=0.75, edgecolor="white")
ax.axvline(observed_ratio, color="black", linewidth=2,
           label=f"Observed ratio = {observed_ratio:.2f}")
ax.axvline(ci_ratio_lower, color="darkgreen", linestyle="--",
           linewidth=2, label="95% bootstrap CI")
ax.axvline(ci_ratio_upper, color="darkgreen", linestyle="--",
           linewidth=2)

ax.set_xlabel("λ₁ / λ₂")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Distribution of λ₁ / λ₂",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

Let's reflect on this:

1. Why is bootstrap especially attractive for a statistic like $\lambda_1 / \lambda_2$?
2. What is the trade-off? What do you gain, and what do you give up, by relying on resampling instead of a closed-form formula?


## Part 6: Model Selection — One Population or Two?

By now you have:
- a visual separation in PC space
- the species labels that explain it
- uncertainty estimates for the PCA quantities

The final question is:

> **Can we justify a two-population model statistically, or could one broad Gaussian population explain the data just as well?**

This is where AIC comes in.

### The idea

A more flexible model nearly always improves the training likelihood. That alone is not enough. AIC asks whether the improvement in fit is worth the increase in complexity.

$$
\mathrm{AIC} = -2 \log L + 2k
$$

where $\log L$ is the total log-likelihood and $k$ is the number of free parameters.

Lower AIC is better.


### The two models

**Model 1: Single Gaussian**

All crabs are drawn from one 5D Gaussian $\mathcal{N}(\mu, \Sigma)$.

**Model 2: Two-component Gaussian mixture**

The population is a mixture of two Gaussians, one for each species:

$$
p(x) = \pi \cdot \mathcal{N}(x \mid \mu_B, \Sigma_B)
+ (1-\pi) \cdot \mathcal{N}(x \mid \mu_O, \Sigma_O)
$$

We'll use the known species labels to fit the two components. That keeps the focus on **model selection**, not on mixture optimization.


### Helpers

We provide the numerically delicate pieces: a stable multivariate Gaussian log-density and a stable `logsumexp` helper for the mixture likelihood.


In [ ]:
def mvn_log_pdf(X, mu, Sigma):
    """Compute log-density of a multivariate Normal for each row."""
    p = len(mu)
    X_centered = X - mu

    # small regularization for numerical stability
    Sigma_reg = Sigma + 1e-6 * np.eye(p)

    sign, log_det = np.linalg.slogdet(Sigma_reg)
    Sigma_inv = np.linalg.inv(Sigma_reg)
    quad = np.sum((X_centered @ Sigma_inv) * X_centered, axis=1)

    return -0.5 * (p * np.log(2 * np.pi) + log_det + quad)


def logsumexp(a, b):
    """Stable computation of log(exp(a) + exp(b))."""
    m = np.maximum(a, b)
    return m + np.log(np.exp(a - m) + np.exp(b - m))

### TODO 6a

Fit **Model 1** by computing the MLE mean and covariance for all crabs.

For likelihood-based model fitting, use the MLE covariance with denominator $n$ rather than $n-1$.


In [ ]:
# TODO 6a: Fit Model 1
mu_all = ...
X_all_centered = ...
Sigma_all = ...

print("Mean vector for Model 1:")
print(np.round(mu_all, 3))
print("\nCovariance matrix for Model 1:")
print(np.round(Sigma_all, 2))

### TODO 6b

Compute the total log-likelihood of all 200 crabs under Model 1.


In [ ]:
# TODO 6b: Log-likelihood for Model 1
log_lik_model1 = ...

print(f"Log-likelihood (Model 1): {log_lik_model1:.2f}")

### TODO 6c

Count the free parameters in Model 1.

For a 5D Gaussian:
- mean vector: 5 parameters
- symmetric covariance matrix: $5(5+1)/2 = 15$ parameters

How many total?


In [ ]:
# TODO 6c: Parameter count for Model 1
k_model1 = ...

print(f"Parameters in Model 1: {k_model1}")

### TODO 6d

Fit **Model 2** by separating the data by species and fitting a Gaussian to each group. Also estimate the mixing weight $\pi$ as the fraction of Blue crabs.


In [ ]:
# TODO 6d: Fit Model 2
mask_blue = ...
mask_orange = ...

X_blue = ...
X_orange = ...

mu_blue = ...
X_blue_centered = ...
Sigma_blue = ...

mu_orange = ...
X_orange_centered = ...
Sigma_orange = ...

pi_blue = ...

print(f"Blue crabs:   n = {mask_blue.sum()}, π = {pi_blue:.3f}")
print(f"Orange crabs: n = {mask_orange.sum()}, π = {1-pi_blue:.3f}")

### TODO 6e

Compute the total log-likelihood under Model 2.

For each crab, the mixture log-density is

$$
\log p(x) = \log \left(
\pi e^{\log f_B(x)} + (1-\pi)e^{\log f_O(x)}
\right)
$$

Use `logsumexp` together with the component log-densities.


In [ ]:
# TODO 6e: Log-likelihood for Model 2
log_prob_blue = ...
log_prob_orange = ...

log_weighted_blue = ...
log_weighted_orange = ...

log_mixture = ...
log_lik_model2 = ...

print(f"Log-likelihood (Model 2): {log_lik_model2:.2f}")

### TODO 6f

Count the free parameters in Model 2.

- Blue component: mean (5) + covariance (15) = 20
- Orange component: mean (5) + covariance (15) = 20
- mixing proportion: 1 free parameter

How many total?


In [ ]:
# TODO 6f: Parameter count for Model 2
k_model2 = ...

print(f"Parameters in Model 2: {k_model2}")

### TODO 6g

Compute AIC for both models and decide which one is preferred.


In [ ]:
# TODO 6g: AIC comparison
AIC_model1 = ...
AIC_model2 = ...

print("Model 1: Single Gaussian")
print(f"  log-likelihood = {log_lik_model1:.2f}")
print(f"  parameters     = {k_model1}")
print(f"  AIC            = {AIC_model1:.2f}")
print()
print("Model 2: Two-component mixture")
print(f"  log-likelihood = {log_lik_model2:.2f}")
print(f"  parameters     = {k_model2}")
print(f"  AIC            = {AIC_model2:.2f}")
print()
print(f"ΔAIC (Model 1 - Model 2): {AIC_model1 - AIC_model2:.2f}")

In [ ]:
# Provided: visualize model means in PC space
mu_all_proj = (mu_all - X_mean) @ V2
mu_blue_proj = (mu_blue - X_mean) @ V2
mu_orange_proj = (mu_orange - X_mean) @ V2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model 1
for sp, color, label in [("B", "dodgerblue", "Blue"),
                         ("O", "darkorange", "Orange")]:
    mask = crabs["sp"] == sp
    axes[0].scatter(Z[mask, 0], Z[mask, 1],
                    c=color, alpha=0.4, s=30,
                    edgecolor="white", linewidth=0.3,
                    label=label)
axes[0].scatter(mu_all_proj[0], mu_all_proj[1],
                c="black", s=300, marker="X",
                edgecolor="white", linewidth=2,
                label="Single Gaussian mean", zorder=10)
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].set_title("Model 1: Single Gaussian",
                  fontsize=13, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Model 2
for sp, color, label in [("B", "dodgerblue", "Blue"),
                         ("O", "darkorange", "Orange")]:
    mask = crabs["sp"] == sp
    axes[1].scatter(Z[mask, 0], Z[mask, 1],
                    c=color, alpha=0.4, s=30,
                    edgecolor="white", linewidth=0.3,
                    label=label)
axes[1].scatter(mu_blue_proj[0], mu_blue_proj[1],
                c="dodgerblue", s=300, marker="X",
                edgecolor="white", linewidth=2,
                label="Blue mean", zorder=10)
axes[1].scatter(mu_orange_proj[0], mu_orange_proj[1],
                c="darkorange", s=300, marker="X",
                edgecolor="white", linewidth=2,
                label="Orange mean", zorder=10)
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].set_title("Model 2: Two Gaussian Components",
                  fontsize=13, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle("Model Comparison in PC Space",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

Let's reflect on this:

1. Why isn't a larger log-likelihood automatically enough to choose the better model?
2. Model 2 has many more parameters. Did the improvement in fit outweigh the penalty?
3. What evidence now supports the claim that the crab data is better modeled as two populations rather than one?

---


## Report Your Results

Run the cell below and enter the values on the course platform. Round as indicated.


In [ ]:
print("=" * 60)
print("MODULE 4 REPORT VALUES")
print("=" * 60)
print(f"R1.  PVE for PC1 (4 dec):                  "
      f"{pve[0]:.4f}")
print(f"R2.  Cumulative PVE, first 2 PCs (4 dec):  "
      f"{cpve[1]:.4f}")
print(f"R3.  PCs needed for 95% variance:          "
      f"{pcs_for_95}")
print(f"R4.  Bootstrap SE of PVE PC1 (4 dec):      "
      f"{boot_se_pve1:.4f}")
print(f"R5.  95% Normal CI mean CL, lower:         "
      f"{ci_mean_normal_lower:.3f}")
print(f"R6.  95% Normal CI mean CL, upper:         "
      f"{ci_mean_normal_upper:.3f}")
print(f"R7.  95% Boot CI PVE PC1, lower (4 dec):   "
      f"{ci_pve_boot_lower:.4f}")
print(f"R8.  95% Boot CI PVE PC1, upper (4 dec):   "
      f"{ci_pve_boot_upper:.4f}")
print(f"R9.  Parameters in Model 2:                "
      f"{k_model2}")
print(f"R10. ΔAIC (Model 1 - Model 2) (2 dec):     "
      f"{AIC_model1 - AIC_model2:.2f}")

## Reflection Questions

Discuss these with the course chatbot or on the discussion forum.


**Q1.** In this lab, PCA revealed species as the dominant source of variation. What exactly does it mean, geometrically, for species separation to align with PC1?


**Q2.** You bootstrapped both a simple statistic (the mean of CL) and a PCA statistic (PVE of PC1). What stayed the same across both procedures, and what changed?


**Q3.** Why is the Normal-approximation confidence interval reasonable for the mean of CL, but more questionable for PVE or for the ratio $\lambda_1 / \lambda_2$?


**Q4.** In the AIC comparison, Model 2 almost certainly has a higher likelihood than Model 1. Why doesn't that alone settle the question? What role does the penalty term play?


**Q5.** We fit the two Gaussian components using the known species labels. If those labels were hidden, what would become harder about the model-fitting step? Which part of the lab would stay conceptually the same?


**Q6.** In this lab, PCA used the raw covariance matrix. What would change if you standardized each variable first and did PCA on the correlation matrix instead? When would that be a better choice?


**Q7.** Looking back across the course: Module 1 hinted at bimodality in the FL/CL ratio, Module 2 found sex effects, Module 3 exposed unexplained banding, and Module 4 revealed species through PCA. At what point did the evidence for a hidden variable become convincing to you?
